In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt


In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)


Using device: cuda


load the Dataset

In [3]:
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor()
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=False, transform=transform
)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=False, transform=transform
)

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True, num_workers=2
)

testloader = torch.utils.data.DataLoader(
    testset, batch_size=128, shuffle=False, num_workers=2
)


In [9]:
import torch.nn as nn
import torch.nn.functional as F

class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.AvgPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)

        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = x.view(-1, 16*5*5)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x


In [10]:
model = LeNet5().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [12]:
from tqdm import tqdm
import time

start = time.time()

epochs = 10

for epoch in range(epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    loop = tqdm(trainloader, desc=f"Epoch [{epoch+1}/{epochs}]")

    for images, labels in loop:

        # MOVE DATA TO GPU
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # UPDATE PROGRESS BAR
        loop.set_postfix(
            loss=running_loss/total,
            acc=100*correct/total
        )

    print(f"Epoch {epoch+1} Training Accuracy: {100*correct/total:.2f}%")

end = time.time()
print("Training Time:", end - start)


Epoch [1/10]: 100%|██████████| 391/391 [00:17<00:00, 22.47it/s, acc=29.7, loss=0.015]  


Epoch 1 Training Accuracy: 29.69%


Epoch [2/10]: 100%|██████████| 391/391 [00:20<00:00, 19.34it/s, acc=41.9, loss=0.0126]


Epoch 2 Training Accuracy: 41.89%


Epoch [3/10]: 100%|██████████| 391/391 [00:31<00:00, 12.41it/s, acc=46.4, loss=0.0117] 


...

Epoch 8 Training Accuracy: 54.40%


Epoch [9/10]: 100%|██████████| 391/391 [00:15<00:00, 24.56it/s, acc=55.4, loss=0.00979] 


Epoch 9 Training Accuracy: 55.37%


Epoch [10/10]: 100%|██████████| 391/391 [00:16<00:00, 24.17it/s, acc=56.2, loss=0.0096]  

Epoch 10 Training Accuracy: 56.17%
Training Time: 185.47236275672913


TEST ACCURACY (GPU)

In [13]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Test Accuracy:", 100 * correct / total)


Test Accuracy: 54.78


# training function

In [7]:
from tqdm import tqdm
import torch
import time

def train_model(model, trainloader, criterion, optimizer, epochs, device):

    model.to(device)
    start_time = time.time()

    for epoch in range(epochs):

        model.train()

        running_loss = 0
        correct = 0
        total = 0

        loop = tqdm(trainloader, desc=f"Epoch [{epoch+1}/{epochs}]")

        for images, labels in loop:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            loop.set_postfix(
                loss=running_loss/total,
                acc=100*correct/total
            )

        print(f"Epoch {epoch+1} Training Accuracy: {100*correct/total:.2f}%")

    print("Training Time:", time.time() - start_time)


# testing function

In [8]:
def test_model(model, testloader, device):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in testloader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Test Accuracy:", 100 * correct / total)


# alexnet

In [29]:
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((227, 227)),   # ⭐ ONLY CHANGE
    transforms.ToTensor()
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=False, transform=transform
)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=False, transform=transform
)

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True, num_workers=2
)

testloader = torch.utils.data.DataLoader(
    testset, batch_size=128, shuffle=False, num_workers=2
)


In [15]:
import torch.nn as nn

class AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super(AlexNet, self).__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Dropout(),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),

            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),

            nn.Linear(4096, num_classes)
        )

    def forward(self, x):

        x = self.features(x)

        x = x.view(x.size(0), 256 * 6 * 6)

        x = self.classifier(x)

        return x


In [20]:
model = AlexNet()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

train_model(model, trainloader, criterion, optimizer, epochs=10, device=device)
test_model(model, testloader, device)


Epoch [1/10]: 100%|██████████| 391/391 [02:03<00:00,  3.17it/s, acc=22, loss=0.0162]  


Epoch 1 Training Accuracy: 22.03%


Epoch [2/10]: 100%|██████████| 391/391 [02:04<00:00,  3.15it/s, acc=40.3, loss=0.0127]


Epoch 2 Training Accuracy: 40.35%


Epoch [3/10]: 100%|██████████| 391/391 [02:21<00:00,  2.76it/s, acc=48.8, loss=0.011] 


...

Epoch 8 Training Accuracy: 67.94%


Epoch [9/10]: 100%|██████████| 391/391 [01:55<00:00,  3.38it/s, acc=69.9, loss=0.00671]


Epoch 9 Training Accuracy: 69.88%


Epoch [10/10]: 100%|██████████| 391/391 [02:28<00:00,  2.63it/s, acc=71.4, loss=0.00634]


Epoch 10 Training Accuracy: 71.43%
Training Time: 1699.9991307258606
Test Accuracy: 69.72


# VGG16

In [4]:
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),   # reduced size
    transforms.ToTensor()
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=False, transform=transform
)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=False, transform=transform
)

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=8, shuffle=True, num_workers=2   # reduced batch size
)

testloader = torch.utils.data.DataLoader(
    testset, batch_size=8, shuffle=False, num_workers=2
)


In [5]:
import torch.nn as nn

class VGG16(nn.Module):
    def __init__(self, num_classes=10):
        super(VGG16, self).__init__()

        self.features = nn.Sequential(

            # Block 1
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 2
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 3
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 4
            nn.Conv2d(256, 512, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 5
            nn.Conv2d(512, 512, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(True),
            nn.Dropout(),

            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),

            nn.Linear(4096, num_classes)
        )

    def forward(self, x):

        x = self.features(x)

        x = x.view(x.size(0), -1)

        x = self.classifier(x)

        return x


In [24]:
torch.cuda.empty_cache()
